# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlishaYaqub/FlyRank-ML-internship-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a scoring task. The decision I'm supporting is "which pages should be reviewed first," not a single yes/no call or label per page instead its a ranked list. So the output is a priority score per page, not a category. Classification is a building block underneath it (is this page declining or not), but the final task is scoring: rank all pages by how urgently they need review.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

url = "https://raw.githubusercontent.com/AlishaYaqub/FlyRank-ML-internship-week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("trend_direction distribution:")
print(df["trend_direction"].value_counts())

trend_direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My proxy target is whether a page is currently declining: trend_direction == "down". This comes from trend_pct, an observed measured change in search performance, not a rule I defined myself — 16,262 of 30,000 pages (54%) fall into this group, with a median trend of about -56%. I will predict a page's likelihood of being in this group using signals other than trend_direction/trend_pct themselves, since those two columns directly encode the outcome and can't be used as features without leaking the answer.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining"] = df["trend_direction"] == "down"
print(df["is_declining"].value_counts(normalize=True).round(2))
print()
print(df.groupby("trend_direction")["trend_pct"].median())

is_declining
True     0.54
False    0.46
Name: proportion, dtype: float64

trend_direction
down     -55.60
flat        NaN
new         NaN
stable    -3.80
up        62.55
Name: trend_pct, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric is precision@K: of the top K pages my score ranks as most urgent, what fraction are actually declining (is_declining == True)? I'm choosing K=100 (a realistic weekly review batch for an editor). This is a defensible number because it's directly about the decision — not overall accuracy, which doesn't tell an editor whether the pages at the top of their queue are worth their time.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Baseline precision@100 using a naive single-signal rule (rank by worst position),
# to give a number "good" will need to beat once a real score is built.
# avg_position == 0 means "no data" per the data dictionary, so exclude those first.
K = 100
valid = df[df["avg_position"] > 0]
naive_rank = valid.sort_values("avg_position", ascending=False).head(K)
baseline_precision = naive_rank["is_declining"].mean()
print(f"Naive baseline precision@{K}: {baseline_precision:.2f}")

Naive baseline precision@100: 0.19


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (one page), identified by content_id. The dataset has 30,000 rows and 30,000 unique content_id values, confirming no duplicates — each page appears exactly once with its trailing-90-day metrics.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Rows:", len(df), "| Unique content_id:", df["content_id"].nunique())
df[["content_id", "client_id", "content_type", "avg_position", "ctr",
    "trend_direction", "trend_pct"]].head()

Rows: 30000 | Unique content_id: 30000


,content_id,client_id,content_type,avg_position,ctr,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,0.76,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,0.05,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,0.09,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,0.49,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,0.13,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single-signal rule doesn't cleanly separate declining pages. Looking at position_tier alone, decline rates range from 24% (top_3) to 61% (striking) across tiers, but they don't move in one clean direction — "deep" position pages decline less (34%) than "page_1" pages (57%). impression_tier is even messier: "low" impressions decline at 45%, but "moderate" impressions decline more, at 61%. A fixed if-statement threshold on any one column would misclassify a large share of pages either way. ML earns its place because the real pattern likely comes from combining several imperfect, sometimes-contradictory signals (position, impressions, freshness, engagement) rather than any one of them alone.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Decline rate by position_tier:")
print(pd.crosstab(df["position_tier"], df["is_declining"], normalize="index").round(2))
print()
print("Decline rate by impression_tier:")
print(pd.crosstab(df["impression_tier"], df["is_declining"], normalize="index").round(2))

Decline rate by position_tier:
is_declining   False  True 
position_tier              
deep            0.66   0.34
page_1          0.43   0.57
page_3_5        0.44   0.56
striking        0.39   0.61
top_3           0.76   0.24

Decline rate by impression_tier:
is_declining     False  True 
impression_tier              
excellent         0.54   0.46
good              0.41   0.59
low               0.55   0.45
moderate          0.39   0.61


## Self-check

Before you submit, confirm each line honestly:

- [YES] Every section above is filled — markdown thinking AND the code that backs it  
- [YES] The notebook runs top to bottom with no errors (Runtime → Run all)  
- [YES] No client names, URLs, or private queries anywhere
- [YES] My claims use careful words: observed, measured, directional, decision-support
- [YES] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.